# Backtesting Avanzado — Notebook Exploratorio
## Proyecto de Investigacion Aplicada en Mercados Financieros
**Equipo 1 | Primavera 2026 | Tec de Monterrey**

Este notebook presenta el analisis exploratorio de indicadores tecnicos y estrategias de backtesting sobre las 7 Magnificas y empresas industriales seleccionadas.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime

from indicators import sma, ema, rsi, macd, bollinger_bands
from strategies import STRATEGY_REGISTRY
from engine import BacktestEngine, compute_metrics

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

## 1. Carga de Datos

In [ ]:
TICKERS = ['NVDA', 'AAPL', 'MSFT', 'META', 'GOOGL', 'TSLA', 'AMZN', 'GE', 'HD']
START = '2022-01-01'
END = datetime.now().strftime('%Y-%m-%d')

data = {}
for t in TICKERS:
    df = yf.download(t, start=START, end=END, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    data[t] = df[['Open','High','Low','Close','Volume']].dropna()

print(f'Tickers cargados: {len(data)}')
for t, df in data.items():
    print(f'  {t}: {len(df)} bars, {df.index[0].date()} -> {df.index[-1].date()}')

## 2. Indicadores Tecnicos — Ejemplo NVDA

In [ ]:
ticker = 'NVDA'
df = data[ticker]
close = df['Close']

ind = pd.DataFrame(index=df.index)
ind['Close'] = close
ind['SMA_20'] = sma(close, 20)
ind['SMA_50'] = sma(close, 50)
ind['EMA_12'] = ema(close, 12)
ind['EMA_26'] = ema(close, 26)
ind['RSI_14'] = rsi(close, 14)
ind['MACD'], ind['MACD_Signal'], ind['MACD_Hist'] = macd(close)
ind['BB_Upper'], ind['BB_Mid'], ind['BB_Lower'], ind['BB_PctB'], ind['BB_Width'] = bollinger_bands(close)

ind.tail(10)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True)

ax = axes[0]
ax.plot(ind.index, ind['Close'], 'k-', lw=0.8, label='Close')
ax.plot(ind.index, ind['SMA_20'], color='#2196F3', lw=0.7, label='SMA(20)')
ax.plot(ind.index, ind['SMA_50'], color='#FF9800', lw=0.7, label='SMA(50)')
ax.fill_between(ind.index, ind['BB_Upper'], ind['BB_Lower'], alpha=0.08, color='purple')
ax.set_title(f'{ticker} — Price + Moving Averages + Bollinger Bands', fontweight='bold')
ax.legend(loc='upper left', fontsize=8)

ax = axes[1]
ax.plot(ind.index, ind['RSI_14'], color='#9C27B0', lw=0.8)
ax.axhline(70, color='r', ls='--', lw=0.5); ax.axhline(30, color='g', ls='--', lw=0.5)
ax.set_title('RSI(14)', fontweight='bold'); ax.set_ylim(0,100)

ax = axes[2]
ax.plot(ind.index, ind['MACD'], color='#2196F3', lw=0.8, label='MACD')
ax.plot(ind.index, ind['MACD_Signal'], color='#FF9800', lw=0.8, label='Signal')
colors_h = ['#4CAF50' if v>=0 else '#F44336' for v in ind['MACD_Hist']]
ax.bar(ind.index, ind['MACD_Hist'], color=colors_h, alpha=0.5, width=1.5)
ax.set_title('MACD(12,26,9)', fontweight='bold'); ax.legend(fontsize=8)

ax = axes[3]
ax.plot(ind.index, ind['BB_PctB'], color='#673AB7', lw=0.8)
ax.axhline(1, color='r', ls='--', lw=0.5); ax.axhline(0, color='g', ls='--', lw=0.5)
ax.set_title('Bollinger %B', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Backtesting — Todas las Estrategias

In [ ]:
engine = BacktestEngine(initial_capital=100_000, commission=0.001)

results = []
for t in TICKERS:
    df_t = data[t]
    for name, func in STRATEGY_REGISTRY.items():
        signals = func(df_t)
        portfolio = engine.run(df_t, signals, t)
        eq = np.array(portfolio.equity_curve)
        m = compute_metrics(eq, portfolio.trades)
        m['ticker'] = t
        m['strategy'] = name
        results.append(m)

results_df = pd.DataFrame(results)
results_df = results_df[['ticker','strategy','total_return_pct','sharpe_ratio','sortino_ratio','max_drawdown_pct','win_rate_pct','total_trades','final_equity']]
print(f'Total combinaciones: {len(results_df)}')
results_df.sort_values('sharpe_ratio', ascending=False).head(20)

## 4. Comparativo Individual vs Combinadas

In [ ]:
individual = ['SMA_Crossover','RSI_MeanReversion','MACD_Crossover','Bollinger_Bounce','Bollinger_Squeeze']
combined = ['Combined_MACD_RSI','Combined_SMA_BB','Combined_Triple']

ind_df = results_df[results_df['strategy'].isin(individual)]
comb_df = results_df[results_df['strategy'].isin(combined)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, label in zip(axes, ['sharpe_ratio','win_rate_pct','max_drawdown_pct'], ['Sharpe','Win Rate (%)','Max Drawdown (%)']):
    bp = ax.boxplot([ind_df[col].dropna(), comb_df[col].dropna()], labels=['Individual','Combined'], patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('#90CAF9'); bp['boxes'][1].set_facecolor('#A5D6A7')
    ax.set_title(label, fontweight='bold'); ax.grid(True, alpha=0.3)
plt.suptitle('Individual vs Combined Strategies', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Heatmap de Metricas

In [ ]:
pivot = results_df.pivot_table(index='ticker', columns='strategy', values='sharpe_ratio')

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', interpolation='nearest')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([c.replace('_','\n') for c in pivot.columns], fontsize=7, rotation=45, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i,j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.7)
ax.set_title('Sharpe Ratio Heatmap — All Tickers x Strategies', fontweight='bold')
plt.tight_layout()
plt.show()